# Drex 2.0: decisions for your agent in one API call

Drex is not a chat model. You send it some data (`state`) and a few typed questions, and it returns a **probability for every possible answer**. Nothing is generated, so there's nothing to parse.

That makes it useful for the step in an agent or workflow where you need to *decide*: route this, approve that, escalate or not.

In this notebook:

1. Setup
2. First call
3. The three question types
4. Using the answer to decide
5. Running it on a batch of tickets

> Disclosure: this cookbook is part of a sponsored collaboration with Nace AI. The code and opinions are my own.

## 1. Setup

Create an API key at [console.nace.ai](https://console.nace.ai) and expose it as `NACE_API_KEY` (in your shell or a `.env` file). New accounts get $5 of balance; this notebook uses a fraction of a cent.

In [1]:
import os, json
import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

client = httpx.Client(
    base_url=os.getenv("DREX_BASE_URL", "https://console.nace.ai"),
    headers={"Authorization": f"Bearer {os.environ['NACE_API_KEY']}"},
    timeout=60,
)

def decide(state, questions, model="drex-v2.0"):
    r = client.post("/v1/systemone", json={"model": model, "state": state, "questions": questions})
    r.raise_for_status()
    return r.json()

That's the whole client. The API is a single endpoint, `POST /v1/systemone`, with plain JSON.

## 2. First call

One piece of text, one yes/no question:

In [2]:
res = decide(
    "Help! My payouts have been failing for 3 days.",
    {"is_urgent": {"type": "noul", "instructions": "Does this convey urgency?"}},
)
print(json.dumps(res, indent=2))

{
  "model": "drex-v2.0",
  "answers": {
    "is_urgent": {
      "type": "noul",
      "noul": 0.7353,
      "confidence": 0.7353
    }
  },
  "usage": {
    "input_tokens": 26,
    "output_tokens": 24
  },
  "evaluation_time_ms": 51.173,
  "request_id": "req_5a4296f33b586afdd786b96cd64dc932"
}


- `noul` is the probability that the answer is **yes** (0 to 1).
- `usage.input_tokens` is what you pay for. Output tokens are free.
- `evaluation_time_ms` is how long the model took on Nace's side.

## 3. The three question types

| Type | Use it for | You get back |
|---|---|---|
| `noul` | Yes / no | Probability of yes |
| `choice` | One label from a set | The top label + a probability per label |
| `score` | A position on an ordered scale | The expected level (can be fractional) |

You can ask all of them about the same text in **one request**:

In [3]:
QUESTIONS = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this support ticket?",
        "criteria": {
            "billing": "Payments, invoices, charges, refunds, plans and pricing",
            "technical": "Bugs, errors, outages, performance and integrations",
            "account": "Login, access, users, permissions and account settings",
            "other": "Anything else",
        },
    },
    "wants_refund": {
        "type": "noul",
        "instructions": "Is the customer asking for money back?",
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket?",
        "criteria": ["Routine", "Soon", "Urgent"],
    },
}

res = decide("I was charged twice for my October invoice. Please refund the duplicate charge.", QUESTIONS)
print(json.dumps(res["answers"], indent=2))

{
  "team": {
    "type": "choice",
    "choice": "billing",
    "confidence": 0.9019,
    "probabilities": {
      "billing": 0.9264,
      "technical": 0.0204,
      "account": 0.0188,
      "other": 0.0343
    }
  },
  "wants_refund": {
    "type": "noul",
    "noul": 0.8642,
    "confidence": 0.8642
  },
  "urgency": {
    "type": "score",
    "score": 0.9273,
    "legend": [
      "Routine",
      "Soon",
      "Urgent"
    ],
    "probabilities": {
      "0": 0.3964,
      "1": 0.2799,
      "2": 0.3237
    },
    "confidence": 0.5363
  }
}


The `choice` answer includes every label's probability, not just the winner. If the top two are close, the model is telling you it isn't sure.

## 4. Using the answer to decide

Drex doesn't make the decision for you: it gives you probabilities and your code sets the rules. A common pattern is to act automatically when the model is confident and send the rest to a person:

In [4]:
def handle(ticket):
    a = decide(ticket, QUESTIONS)["answers"]
    team = a["team"]["choice"]
    confidence = a["team"]["probabilities"][team]

    if confidence < 0.6:
        return "send to a human"

    steps = [f"route to {team}"]
    if a["wants_refund"]["noul"] >= 0.5:
        steps.append("open refund flow")
    if a["urgency"]["score"] >= 1.5:
        steps.append("page on-call")
    return " + ".join(steps)

print(handle("Your API has been returning 500 errors for the last 2 hours. Our checkout is down."))
print(handle("hi"))

route to technical + page on-call


send to a human


The thresholds (`0.6`, `0.5`, `1.5`) are yours to choose. Run a few real examples, look at the probabilities you get, and set them from there. Different questions can need different thresholds.

## 5. A batch of tickets

In [5]:
tickets = [
    "I was charged twice for my October invoice. Please refund the duplicate charge.",
    "Your API has been returning 500 errors for the last 2 hours. Our checkout is down.",
    "How do I change the email address on my account?",
    "Me cobraron dos veces la suscripción de este mes. Necesito que me devuelvan uno de los cobros.",
    "I'm not asking for money back, I just want the duplicate charge to stop next month.",
    "hi",
]

rows = []
for t in tickets:
    res = decide(t, QUESTIONS)
    a = res["answers"]
    rows.append({
        "ticket": t[:60],
        "team": a["team"]["choice"],
        "team_p": round(a["team"]["probabilities"][a["team"]["choice"]], 2),
        "refund_p": round(a["wants_refund"]["noul"], 2),
        "urgency": round(a["urgency"]["score"], 2),
        "model_ms": res.get("evaluation_time_ms"),
        "tokens": res["usage"]["input_tokens"],
    })

pd.DataFrame(rows)

,ticket,team,team_p,refund_p,urgency,model_ms,tokens
0,I was charged twice for my October invoice. Pl...,billing,0.93,0.86,0.93,17.011,111
1,Your API has been returning 500 errors for the...,technical,0.92,0.10,1.81,17.768,118
2,How do I change the email address on my account?,account,0.69,0.03,0.66,15.839,107
3,Me cobraron dos veces la suscripción de este m...,billing,0.95,0.91,1.00,15.294,122
4,"I'm not asking for money back, I just want the...",billing,0.85,0.08,0.81,14.517,115
5,hi,other,0.36,0.39,1.14,99.887,97


Things to look at in the table:

- The Spanish ticket goes through the same questions written in English.
- "I'm **not** asking for money back" should get a low `refund_p`.
- A vague "hi" should come back with low confidence, which is exactly when your flow should hand it to a person.

## Wrap-up

- One endpoint, plain JSON, no output to parse.
- Ask several questions in one call: you pay for the text once.
- You get probabilities, so *you* decide when the agent acts and when a human does.

Docs: [drex.nace.ai/docs](https://drex.nace.ai/docs) · Models: [github.com/nace-ai/drex-decision-models](https://github.com/nace-ai/drex-decision-models)